In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# RBFE on a ligand set (workflow)

End-to-end workflow for a congeneric series:

1. Load the BRD protein and three ligands, register them on the data platform
2. Submit `RBFE(protein=..., ligands=...)` — Konnektor plans pairs, then system-prep and FEP run per edge
3. Quote, confirm, and watch progress
4. Inspect prepared systems and ΔΔG results

## Setup

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Ligand,
    LigandSet,
    Protein,
    RBFE,
    RBFEParams,
)
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

## 1. Load structures and register on the data platform

We use the BRD4 example protein and three congeneric ligands from the bundled
dataset. `sync()` uploads files (when needed) and registers records on the
data platform. Konnektor mode requires every ligand to have a platform `id` and
remote `file_path`.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync(client=client)
protein.id

In [ ]:
ligands = []
for ligand_file in ["brd-2.sdf", "brd-3.sdf", "brd-4.sdf"]:
    ligand = Ligand.from_sdf(BRD_DATA_DIR / ligand_file)
    ligand.sync(client=client)
    ligands.append(ligand)

ligand_set = LigandSet(ligands=ligands)
ligand_set

## 2. Submit RBFE workflow (Konnektor → system-prep → FEP)

Pass the ligand set (or a plain list). The platform infers
`steps=["konnektor", "system-prep", "rbfe"]`. Quote first, then confirm to
start the job.

`test_run=1` shortens the simulation for exploration; use `test_run=0` for
production-quality results.

## 3. Quote, confirm, and watch

In [ ]:
rbfe = RBFE(
    protein=protein,
    ligands=ligand_set,
    network_type="mst",
    params=RBFEParams(test_run=1),
    client=client,
)
assert rbfe.steps == ["konnektor", "system-prep", "rbfe"]
rbfe

In [ ]:
rbfe.start(quote=True)
rbfe.estimate

In [ ]:
rbfe.confirm()

In [ ]:
task = await rbfe.watch()

In [ ]:
rbfe.sync()
rbfe.progress

## Results

In [ ]:
rbfe.get_results()

In [ ]:
rbfe.get_user_logs()

In [ ]:
# Inspect one prepared system after system-prep completes (pick any edge)
rbfe.get_prepared_system(
    ligand1_id=ligands[0].id,
    ligand2_id=ligands[1].id,
)

In [ ]:
[ligand.id for ligand in ligands]

In [ ]:
rbfe.cancel()

In [ ]:
rbfe.id

In [ ]:
# Optional: reload the most recent RBFE execution from this session
rbfe = RBFE.from_last_run(client=client)
system = rbfe.get_prepared_system(
    ligand1_id=ligands[0].id,
    ligand2_id=ligands[1].id,
)
system.show(solute=False)

In [ ]:
rbfe = RBFE.from_id("b040475a-ab7b-4b59-bd5e-4d41dae7df7c")
task = await rbfe.watch()

In [ ]:
rbfe.dto